In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ==============================================================================
# --- 1. 配置 (您需要在这里指定要评估哪个模型) ---
# ==============================================================================
# SAM2 Hydra 配置名
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"

# !! 修改这里，指向您在ISIC上微调后的模型检查点 !!
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/CVC_bLoss2.5_adapter_run1/checkpoints/checkpoint.pt"

# Hydra Overrides, 用于加载带Adapter的模型结构
HYDRA_OVERRIDES = [
    "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
    "+model.image_encoder.trunk.use_adapter=True",
    "+model.use_adapter=True",
]

# ISIC数据集的路径
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/CVC-ClinicDB_for_SAM2"
# 使用测试集 val.txt 进行评估
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets","val.txt")


# ==============================================================================
# --- 2. 指标计算函数 (全面升级版) ---
# ==============================================================================
def calculate_metrics(gt_mask, pred_mask):
    """
    计算 Dice, IoU, Accuracy, Sensitivity (Recall), 和 Specificity.
    """
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    
    # 计算 TP, FP, TN, FN
    tp = np.logical_and(pred_mask_bool, gt_mask_bool).sum() # 真阳性
    fp = np.logical_and(pred_mask_bool, ~gt_mask_bool).sum() # 假阳性
    tn = np.logical_and(~pred_mask_bool, ~gt_mask_bool).sum() # 真阴性
    fn = np.logical_and(~pred_mask_bool, gt_mask_bool).sum() # 假阴性

    # --- 计算各项指标 ---
    
    # Dice Score
    dice = (2. * tp) / (2 * tp + fp + fn + 1e-8)
    
    # Intersection over Union (IoU)
    iou = tp / (tp + fp + fn + 1e-8)
    
    # Accuracy: (TP + TN) / (TP + TN + FP + FN)
    # 准确率: 所有像素中，预测正确的比例
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    
    # Sensitivity (Recall): TP / (TP + FN)
    # 敏感性 (召回率): 所有真的病灶像素中，被正确预测出来的比例
    sensitivity = tp / (tp + fn + 1e-8)
    
    # Specificity: TN / (TN + FP)
    # 特异性: 所有真的背景像素中，被正确预测出来的比例
    specificity = tn / (tn + fp + 1e-8)
    
    return dice, iou, accuracy, sensitivity, specificity


# ==============================================================================
# --- 3. 主评估流程 ---
# ==============================================================================
def main():
    # --- 模型加载 ---
    print("Loading SAM2 model in Adapter mode...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = build_sam2(
        SAM2_CFG_NAME, 
        SAM2_CHECKPOINT_PATH, 
        device=device,
        hydra_overrides_extra=HYDRA_OVERRIDES
    )
    predictor = SAM2ImagePredictor(model)
    print(f"Model and Predictor loaded on {device}.")

    # --- 数据集文件读取 ---
    if not os.path.exists(TEST_SET_FILE):
        raise FileNotFoundError(f"Test set file not found: {TEST_SET_FILE}")

    with open(TEST_SET_FILE, 'r') as f:
        test_sample_names = [line.strip() for line in f.readlines()]
    
    print(f"\nFound {len(test_sample_names)} samples in the test set. Starting evaluation...")
    
    results = []
    images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
    annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

    for sample_name in tqdm(test_sample_names, desc="Evaluating Test Set"):
        image_path = os.path.join(images_dir, sample_name, "00000.jpg")
        mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
        
        if not os.path.exists(image_path) or not os.path.exists(mask_path):
            continue

        image = cv2.imread(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        predictor.set_image(image_rgb)
        contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        box_prompt = np.array([[x, y, x + w, y + h]])

        masks, _, _ = predictor.predict(box=box_prompt, multimask_output=False)
        pred_mask = (masks[0] * 255).astype(np.uint8)
        
        # 计算所有五个指标
        dice, iou, acc, sens, spec = calculate_metrics(gt_mask, pred_mask)
        
        results.append({
            "dice": dice, "iou": iou, "accuracy": acc, 
            "sensitivity": sens, "specificity": spec
        })

        predictor.reset_predictor()

    # --- 结果汇总与展示 (已简化，更专业) ---
    if not results:
        print("No images were processed.")
        return

    df = pd.DataFrame(results)
    
    # 计算所有指标的均值和标准差
    summary = df.agg(['mean', 'std']).T # .T 用于转置，让格式更好看
    summary.columns = ['Mean', 'Std'] # 重命名列
    
    print("\n" + "="*60)
    print("--- Evaluation Summary on ISIC Test Set (val.txt) ---")
    print("="*60)
    print(f"Evaluated on {len(df)} images.")
    print("-" * 60)
    # 使用 to_string() 和 float_format 来美化输出
    print(summary.to_string(float_format=lambda x: f"{x:.4f}"))
    print("="*60)

if __name__ == '__main__':
    main()

Loading SAM2 model in Adapter mode...
Model and Predictor loaded on cuda.

Found 123 samples in the test set. Starting evaluation...


Evaluating Test Set: 100%|██████████| 123/123 [00:13<00:00,  8.98it/s]


--- Evaluation Summary on ISIC Test Set (val.txt) ---
Evaluated on 123 images.
------------------------------------------------------------
              Mean    Std
dice        0.9496 0.0389
iou         0.9064 0.0620
accuracy    0.9917 0.0102
sensitivity 0.9565 0.0601
specificity 0.9955 0.0068
